# Ejercicio 2: Escalamiento de Tickets de Soporte Técnico

**Curso:** IS-484 Inteligencia Artificial I  
**Práctica Calificada 001**  
**Alumno:** Aguilar Flores, Crisólogo  
**Ciclo:** 2026-II

---
## Ficha PEAS

| Componente | Descripción |
|---|---|
| **Percepción (S)** | `tiempo_espera_minutos` (int), `nivel_urgencia` ("baja" / "media" / "alta"), `cliente_premium` (bool) |
| **Acciones (A)** | `atender_nivel_1` (soporte básico), `escalar_a_nivel_2` (soporte intermedio), `escalar_a_nivel_3` (soporte crítico/experto) |
| **Entorno (E)** | Mesa de ayuda (help desk) de una empresa de tecnología. Observable (datos conocidos del ticket), dinámico (nuevos tickets llegan continuamente), discreto en urgencia, episódico por ticket |
| **Objetivo** | Resolver incidencias en el menor tiempo posible, priorizando casos críticos y clientes con acuerdo de nivel de servicio (SLA) premium |
| **Medida de desempeño (P)** | Tiempo de resolución promedio, cumplimiento de SLA, satisfacción del cliente (CSAT), tasa de escalamientos innecesarios |

---
## Reglas de Decisión

| Urgencia | Cliente Premium | Tiempo espera | Acción |
|---|---|---|---|
| `alta` | cualquiera | cualquiera | `escalar_a_nivel_3` |
| `media` | `True` | cualquiera | `escalar_a_nivel_2` |
| `media` | `False` | ≥ 30 min | `escalar_a_nivel_2` |
| `media` | `False` | < 30 min | `atender_nivel_1` |
| `baja` | `True` | ≥ 60 min | `escalar_a_nivel_2` |
| `baja` | `True` | < 60 min | `atender_nivel_1` |
| `baja` | `False` | cualquiera | `atender_nivel_1` |

### Justificación de las reglas

Los tickets de urgencia alta representan incidencias críticas (sistema caído, pérdida de datos) que, sin importar el tiempo de espera ni el tipo de cliente, deben llegar inmediatamente a un experto de nivel 3, ya que cada minuto adicional puede implicar pérdidas económicas o de reputación significativas. Para urgencia media, los clientes premium tienen un SLA contractual que obliga a respuesta garantizada, por lo que se escalan directamente a nivel 2; los clientes estándar se escalan solo si superan 30 minutos de espera, umbral razonable antes de que la insatisfacción escale. Para urgencia baja, el tiempo de espera es el único discriminador para clientes premium (umbral de 60 minutos, el doble que en urgencia media, por la menor criticidad), mientras que los clientes estándar se atienden en nivel 1 independientemente, priorizando la disponibilidad del equipo experto para casos más graves.

---
## Código

In [ ]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """
    Agente de escalamiento de tickets de soporte técnico.
    
    Parámetros:
        tiempo_espera_minutos (int): Minutos que lleva esperando el ticket.
        nivel_urgencia (str): 'baja', 'media' o 'alta'.
        cliente_premium (bool): True si el cliente tiene contrato premium.
    
    Retorna:
        tuple: (accion: str, motivo: str)
    """
    nivel_urgencia = nivel_urgencia.strip().lower()
    
    if nivel_urgencia not in ("baja", "media", "alta"):
        return ("atender_nivel_1",
                f"nivel de urgencia '{nivel_urgencia}' no reconocido; se asigna nivel 1 por defecto")
    
    tipo_cliente = "premium" if cliente_premium else "estándar"
    
    # --- Urgencia ALTA ---
    if nivel_urgencia == "alta":
        return ("escalar_a_nivel_3",
                f"urgencia alta: se requiere atención experta inmediata "
                f"(cliente {tipo_cliente}, {tiempo_espera_minutos} min en espera)")
    
    # --- Urgencia MEDIA ---
    if nivel_urgencia == "media":
        if cliente_premium:
            return ("escalar_a_nivel_2",
                    f"urgencia media con cliente premium: SLA obliga escalamiento "
                    f"({tiempo_espera_minutos} min en espera)")
        elif tiempo_espera_minutos >= 30:
            return ("escalar_a_nivel_2",
                    f"urgencia media y tiempo de espera de {tiempo_espera_minutos} min "
                    f"supera el umbral (30 min); se escala a nivel 2")
        else:
            return ("atender_nivel_1",
                    f"urgencia media con {tiempo_espera_minutos} min de espera "
                    f"(por debajo del umbral de 30 min); se resuelve en nivel 1")
    
    # --- Urgencia BAJA ---
    if nivel_urgencia == "baja":
        if cliente_premium and tiempo_espera_minutos >= 60:
            return ("escalar_a_nivel_2",
                    f"urgencia baja pero cliente premium con {tiempo_espera_minutos} min "
                    f"de espera supera el umbral (60 min)")
        else:
            razon_baja = (f"cliente premium con solo {tiempo_espera_minutos} min de espera"
                          if cliente_premium
                          else f"urgencia baja y cliente estándar")
            return ("atender_nivel_1",
                    f"{razon_baja}; se atiende en nivel 1")

print("Función agente_soporte definida correctamente.")

---
## Simulación y Pruebas

Se prueban 6 combinaciones, incluyendo casos donde las condiciones no coinciden entre sí.

In [ ]:
casos_prueba = [
    # (tiempo, urgencia, premium, descripcion)
    (5,  "alta",  False, "Caso 1 — Urgencia alta, estándar, 5 min    → NIVEL 3 inmediato"),
    (10, "alta",  True,  "Caso 2 — Urgencia alta, premium, 10 min    → NIVEL 3 inmediato"),
    (15, "media", True,  "Caso 3 — Urgencia media, premium, 15 min   → NIVEL 2 por SLA"),
    (45, "media", False, "Caso 4 — Urgencia media, estándar, 45 min  → NIVEL 2 por tiempo"),
    (20, "media", False, "Caso 5 — Urgencia media, estándar, 20 min  → NIVEL 1 (bajo umbral)"),
    (90, "baja",  True,  "Caso 6 — Urgencia BAJA, premium, 90 min    → NIVEL 2 (espera alta)"),
    (30, "baja",  False, "Caso 7 — Urgencia baja, estándar, 30 min   → NIVEL 1"),
]

print("=" * 70)
print("SIMULACIÓN DE TICKETS DE SOPORTE TÉCNICO")
print("=" * 70)

for tiempo, urgencia, premium, desc in casos_prueba:
    accion, motivo = agente_soporte(tiempo, urgencia, premium)
    print(f"\n{desc}")
    print(f"  Tiempo: {tiempo} min | Urgencia: {urgencia} | Premium: {premium}")
    print(f"  → Acción : {accion.upper()}")
    print(f"  → Motivo : {motivo}")

print("\n" + "=" * 70)